In [ ]:
"""
================================================================================
MODULE 1: DATA LOADING AND SOCIOECONOMIC DEPRIVATION INDEX CONSTRUCTION
================================================================================

Project: District-Level Multidimensional Socioeconomic Deprivation Analysis
         in Karnataka, India

Purpose: This module loads district-level data from official government sources,
         validates and normalizes indicators, and constructs a composite
         Socioeconomic Deprivation Index (SEDI).

Author: [Your Name]
Date: February 2026
Version: 2.0 (Simplified)

================================================================================
DATA SOURCES AND INDICATORS
================================================================================

This analysis integrates data from three official government sources:

1. CENSUS OF INDIA 2011
   - Literacy_Rate (%)
   - Urbanization_Rate (%)
   - Electricity_Access (%)
   - Safe_Water_Access (%)
   Source: Office of the Registrar General & Census Commissioner, India

2. NITI AAYOG SDG INDIA INDEX 2021-22
   - Education_Score (SDG 4: Quality Education, 0-100 scale)
   - Health_Score (SDG 3: Good Health and Well-being, 0-100 scale)
   - Infrastructure_Score (SDG 9 & 11: Infrastructure & Sustainable Cities, 0-100 scale)
   Source: NITI Aayog, Government of India

3. ECONOMIC SURVEY OF KARNATAKA 2022-23
   - Per_Capita_Income (INR)
   - Unemployment_Rate (%)
   - Healthcare_Facilities_Per_Lakh (facilities per 100,000 population)
   - Road_Density (km per sq.km)
   Source: Planning, Programme Monitoring and Statistics Department,
           Government of Karnataka

================================================================================
METHODOLOGICAL NOTES
================================================================================

DEPRIVATION vs. POVERTY:
This analysis focuses on DEPRIVATION, not poverty estimation. Deprivation refers
to the multidimensional lack of capabilities, services, and opportunities across
economic, education, health, and infrastructure domains. We use proxy indicators
from official sources because direct household-level poverty data (income/consumption)
is not available at the district level.

THEORETICAL FRAMEWORK:
- Sen's Capability Approach (Sen, 1985)
- UNDP Human Development Index methodology
- Alkire-Foster Multidimensional Poverty Index framework
- NITI Aayog SDG Index approach

INDEX CONSTRUCTION:
The Socioeconomic Deprivation Index (SEDI) uses domain-based theoretical weights
rather than data-driven weights to ensure:
1. Interpretability for policymakers
2. Comparability with national/international indices
3. Stability across time periods
4. Transparency in methodology

================================================================================
"""



from google.colab import drive
drive.mount('/content/drive')


INPUT_FILE = '/content/drive/MyDrive/deprivation_analysis/karnataka_districts_dataset.csv'






import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings

warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


class DataLoader:
    """
    Loads district-level data from a single CSV file containing indicators
    from Census 2011, NITI Aayog SDG Index, and Economic Survey of Karnataka.
    """

    # District name standardization mapping
    DISTRICT_NAME_MAPPING = {
        'Bangalore': 'Bengaluru Urban',
        'Bangalore Urban': 'Bengaluru Urban',
        'Bangalore Rural': 'Bengaluru Rural',
        'Bengaluru': 'Bengaluru Urban',
        'Gulbarga': 'Kalaburagi',
        'Bellary': 'Ballari',
        'Belgaum': 'Belagavi',
        'Shimoga': 'Shivamogga',
        'Tumkur': 'Tumakuru',
        'Mysore': 'Mysuru',
        'Bijapur': 'Vijayapura',
    }

    def __init__(self, filepath: str):
        """
        Initialize DataLoader with file path.

        Parameters:
        -----------
        filepath : str
            Path to the CSV file containing district-level data
        """
        self.filepath = filepath
        self.data = None

    def load_data(self) -> pd.DataFrame:
        """
        Load district-level data from CSV file.

        Returns:
        --------
        pd.DataFrame
            Loaded and validated dataset
        """
        print("\n" + "="*80)
        print("SECTION 1.1: DATA LOADING")
        print("="*80)

        try:
            self.data = pd.read_csv(self.filepath)
            print(f"✓ Successfully loaded data from: {self.filepath}")
            print(f"✓ Dataset shape: {self.data.shape[0]} districts × {self.data.shape[1]} variables")

            # Standardize district names
            self.data = self._standardize_district_names(self.data)

            # Display basic information
            print("\nFirst 5 districts:")
            print(self.data.head())

            return self.data

        except FileNotFoundError:
            raise FileNotFoundError(
                f"Data file not found: {self.filepath}\n"
                f"Please ensure the file exists at the specified location."
            )
        except Exception as e:
            raise Exception(f"Error loading data: {str(e)}")

    def _standardize_district_names(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Standardize district names to official nomenclature.

        Parameters:
        -----------
        df : pd.DataFrame
            Input dataframe with 'District' column

        Returns:
        --------
        pd.DataFrame
            Dataframe with standardized district names
        """
        if 'District' in df.columns:
            df['District'] = df['District'].replace(self.DISTRICT_NAME_MAPPING)
            print(f"✓ District names standardized to official nomenclature")

        return df


class DataValidator:
    """
    Validates data quality: missing values, outliers, and logical consistency.
    """

    def __init__(self, data: pd.DataFrame):
        """
        Initialize DataValidator with dataset.

        Parameters:
        -----------
        data : pd.DataFrame
            District-level dataset to validate
        """
        self.data = data.copy()
        self.validation_report = {}

    def validate(self, missing_threshold: float = 0.20,
                 outlier_method: str = 'iqr') -> pd.DataFrame:
        """
        Perform comprehensive data validation.

        Parameters:
        -----------
        missing_threshold : float
            Maximum proportion of missing values allowed (default: 0.20)
        outlier_method : str
            Method for outlier detection: 'iqr' or 'zscore' (default: 'iqr')

        Returns:
        --------
        pd.DataFrame
            Validated and cleaned dataset
        """
        print("\n" + "="*80)
        print("SECTION 1.2: DATA VALIDATION")
        print("="*80)

        # Check missing values
        self._check_missing_values(missing_threshold)

        # Detect outliers (informational - no removal for small sample)
        self._detect_outliers(method=outlier_method)

        # Check logical consistency
        self._check_logical_consistency()

        # Generate summary report
        self._generate_validation_summary()

        return self.data

    def _check_missing_values(self, threshold: float):
        """Check and handle missing values."""
        missing_count = self.data.isnull().sum()
        missing_pct = (missing_count / len(self.data)) * 100

        missing_summary = pd.DataFrame({
            'Missing_Count': missing_count,
            'Missing_Percentage': missing_pct
        })
        missing_summary = missing_summary[missing_summary['Missing_Count'] > 0]

        if len(missing_summary) > 0:
            print("\n⚠ Missing values detected:")
            print(missing_summary)

            # Handle missing values
            for col in missing_summary.index:
                if col == 'District':
                    continue

                missing_prop = missing_summary.loc[col, 'Missing_Percentage'] / 100

                if missing_prop > threshold:
                    print(f"  ✗ Dropping column '{col}' (>{threshold*100}% missing)")
                    self.data.drop(columns=[col], inplace=True)
                else:
                    # Impute with median for numeric columns
                    median_val = self.data[col].median()
                    self.data[col].fillna(median_val, inplace=True)
                    print(f"  ✓ Imputed '{col}' with median: {median_val:.2f}")

            self.validation_report['missing_values'] = missing_summary.to_dict()
        else:
            print("✓ No missing values detected")
            self.validation_report['missing_values'] = "None"

    def _detect_outliers(self, method: str = 'iqr'):
        """
        Detect outliers using IQR or Z-score method.
        For small samples (n≈31), detection is informational only.
        """
        print(f"\n📊 Outlier Detection (Method: {method.upper()}):")
        print("  Note: For small samples (n≈31), outliers are reported but not removed.")

        numeric_cols = self.data.select_dtypes(include=[np.number]).columns
        outlier_summary = {}

        for col in numeric_cols:
            if method == 'iqr':
                Q1 = self.data[col].quantile(0.25)
                Q3 = self.data[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                outliers = self.data[(self.data[col] < lower_bound) |
                                    (self.data[col] > upper_bound)]

            elif method == 'zscore':
                z_scores = np.abs((self.data[col] - self.data[col].mean()) /
                                 self.data[col].std())
                outliers = self.data[z_scores > 3]

            if len(outliers) > 0:
                outlier_districts = outliers['District'].tolist() if 'District' in outliers.columns else []
                print(f"  • {col}: {len(outliers)} outliers detected - {outlier_districts}")
                outlier_summary[col] = len(outliers)

        if not outlier_summary:
            print("  ✓ No significant outliers detected")

        self.validation_report['outliers'] = outlier_summary

    def _check_logical_consistency(self):
        """Check logical consistency of values."""
        print("\n🔍 Logical Consistency Checks:")

        issues = []

        # Check percentage variables (should be 0-100)
        percentage_cols = ['Literacy_Rate', 'Urbanization_Rate', 'Electricity_Access',
                          'Safe_Water_Access', 'Unemployment_Rate']

        for col in percentage_cols:
            if col in self.data.columns:
                invalid = self.data[(self.data[col] < 0) | (self.data[col] > 100)]
                if len(invalid) > 0:
                    issues.append(f"{col}: {len(invalid)} values outside [0,100] range")

        # Check score variables (should be 0-100)
        score_cols = ['Education_Score', 'Health_Score', 'Infrastructure_Score']

        for col in score_cols:
            if col in self.data.columns:
                invalid = self.data[(self.data[col] < 0) | (self.data[col] > 100)]
                if len(invalid) > 0:
                    issues.append(f"{col}: {len(invalid)} values outside [0,100] range")

        # Check non-negative variables
        non_negative_cols = ['Per_Capita_Income', 'Healthcare_Facilities_Per_Lakh',
                            'Road_Density']

        for col in non_negative_cols:
            if col in self.data.columns:
                invalid = self.data[self.data[col] < 0]
                if len(invalid) > 0:
                    issues.append(f"{col}: {len(invalid)} negative values detected")

        if issues:
            print("  ⚠ Issues detected:")
            for issue in issues:
                print(f"    - {issue}")
            self.validation_report['logical_consistency'] = issues
        else:
            print("  ✓ All values within expected ranges")
            self.validation_report['logical_consistency'] = "Passed"

    def _generate_validation_summary(self):
        """Generate and display validation summary."""
        print("\n" + "-"*80)
        print("VALIDATION SUMMARY")
        print("-"*80)
        print(f"Final dataset shape: {self.data.shape}")
        print(f"Districts: {self.data['District'].nunique() if 'District' in self.data.columns else 'N/A'}")
        print(f"Variables: {len(self.data.columns)}")


class IndicatorNormalizer:
    """
    Normalizes indicators using Min-Max normalization with directionality consideration.
    """

    # Indicator directionality classification
    POSITIVE_INDICATORS = [
        'Literacy_Rate', 'Urbanization_Rate', 'Electricity_Access', 'Safe_Water_Access',
        'Per_Capita_Income', 'Healthcare_Facilities_Per_Lakh', 'Road_Density',
        'Education_Score', 'Health_Score', 'Infrastructure_Score'
    ]

    NEGATIVE_INDICATORS = [
        'Unemployment_Rate'
    ]

    def __init__(self, data: pd.DataFrame):
        """
        Initialize IndicatorNormalizer.

        Parameters:
        -----------
        data : pd.DataFrame
            Validated district-level dataset
        """
        self.data = data.copy()
        self.normalization_metadata = {}

    def normalize(self) -> pd.DataFrame:
        """
        Normalize all indicators using Min-Max normalization.

        For POSITIVE indicators (higher value = better):
            normalized = (X - X_min) / (X_max - X_min)

        For NEGATIVE indicators (higher value = worse):
            normalized = 1 - [(X - X_min) / (X_max - X_min)]

        Result: All normalized indicators in [0, 1] where 1 = best (lowest deprivation)

        Returns:
        --------
        pd.DataFrame
            Dataset with normalized indicators (suffix: _norm)
        """
        print("\n" + "="*80)
        print("SECTION 1.3 & 1.4: INDICATOR CLASSIFICATION AND NORMALIZATION")
        print("="*80)

        print("\n📊 Indicator Directionality:")
        print(f"  POSITIVE indicators (higher = better): {len(self.POSITIVE_INDICATORS)}")
        for ind in self.POSITIVE_INDICATORS:
            if ind in self.data.columns:
                print(f"    • {ind}")

        print(f"\n  NEGATIVE indicators (higher = worse): {len(self.NEGATIVE_INDICATORS)}")
        for ind in self.NEGATIVE_INDICATORS:
            if ind in self.data.columns:
                print(f"    • {ind}")

        print("\n🔄 Normalizing indicators (Min-Max method):")

        # Normalize positive indicators
        for col in self.POSITIVE_INDICATORS:
            if col in self.data.columns:
                self._normalize_positive(col)

        # Normalize negative indicators
        for col in self.NEGATIVE_INDICATORS:
            if col in self.data.columns:
                self._normalize_negative(col)

        return self.data

    def _normalize_positive(self, column: str):
        """Normalize positive indicator (higher = better)."""
        min_val = self.data[column].min()
        max_val = self.data[column].max()
        range_val = max_val - min_val

        if range_val == 0:
            self.data[f'{column}_norm'] = 1.0
            print(f"  ⚠ {column}: No variation (all values equal)")
        else:
            self.data[f'{column}_norm'] = (self.data[column] - min_val) / range_val
            print(f"  ✓ {column}: Normalized [min={min_val:.2f}, max={max_val:.2f}]")

        self.normalization_metadata[column] = {
            'direction': 'positive',
            'min': float(min_val),
            'max': float(max_val),
            'range': float(range_val)
        }

    def _normalize_negative(self, column: str):
        """Normalize negative indicator (higher = worse)."""
        min_val = self.data[column].min()
        max_val = self.data[column].max()
        range_val = max_val - min_val

        if range_val == 0:
            self.data[f'{column}_norm'] = 1.0
            print(f"  ⚠ {column}: No variation (all values equal)")
        else:
            # Invert: 1 - normalized value
            self.data[f'{column}_norm'] = 1 - ((self.data[column] - min_val) / range_val)
            print(f"  ✓ {column} (inverted): Normalized [min={min_val:.2f}, max={max_val:.2f}]")

        self.normalization_metadata[column] = {
            'direction': 'negative',
            'min': float(min_val),
            'max': float(max_val),
            'range': float(range_val)
        }


class SEDIConstructor:
    """
    Constructs the Socioeconomic Deprivation Index (SEDI) using domain-based weights.
    """

    # Domain structure and weights (based on theoretical framework)
    DOMAIN_STRUCTURE = {
        'Economic': {
            'weight': 0.30,
            'indicators': ['Per_Capita_Income_norm', 'Unemployment_Rate_norm']
        },
        'Education': {
            'weight': 0.25,
            'indicators': ['Literacy_Rate_norm', 'Education_Score_norm']
        },
        'Health': {
            'weight': 0.20,
            'indicators': ['Healthcare_Facilities_Per_Lakh_norm', 'Health_Score_norm']
        },
        'Infrastructure': {
            'weight': 0.25,
            'indicators': ['Road_Density_norm', 'Electricity_Access_norm',
                          'Urbanization_Rate_norm', 'Infrastructure_Score_norm']
        }
    }

    def __init__(self, data: pd.DataFrame):
        """
        Initialize SEDIConstructor.

        Parameters:
        -----------
        data : pd.DataFrame
            Dataset with normalized indicators
        """
        self.data = data.copy()

    def construct_sedi(self) -> pd.DataFrame:
        """
        Construct the Socioeconomic Deprivation Index (SEDI).

        SEDI = Σ(weight_d × domain_score_d) × 100

        Where:
        - domain_score_d = average of normalized indicators in domain d
        - weight_d = theoretical weight for domain d
        - Range: 0-100 (higher = lower deprivation)

        Returns:
        --------
        pd.DataFrame
            Dataset with SEDI and domain scores
        """
        print("\n" + "="*80)
        print("SECTION 1.5: SEDI CONSTRUCTION")
        print("="*80)

        print("\n📐 Domain-Based Index Construction:")
        print("  Method: Weighted average of normalized domain scores")
        print("  Theoretical justification: Equal consideration of key development dimensions")
        print("\n  Domain Weights:")

        # Calculate domain scores
        for domain, config in self.DOMAIN_STRUCTURE.items():
            weight = config['weight']
            indicators = [ind for ind in config['indicators'] if ind in self.data.columns]

            if indicators:
                # Domain score = average of normalized indicators
                self.data[f'{domain}_Score'] = self.data[indicators].mean(axis=1)
                print(f"    • {domain}: {weight*100:.0f}% ({len(indicators)} indicators)")
            else:
                print(f"    ⚠ {domain}: No indicators available")

        # Calculate SEDI
        sedi_components = []
        for domain, config in self.DOMAIN_STRUCTURE.items():
            if f'{domain}_Score' in self.data.columns:
                weighted_score = self.data[f'{domain}_Score'] * config['weight']
                sedi_components.append(weighted_score)

        if sedi_components:
            self.data['SEDI'] = sum(sedi_components) * 100  # Scale to 0-100

            print(f"\n✓ SEDI calculated successfully")
            print(f"  Range: {self.data['SEDI'].min():.2f} - {self.data['SEDI'].max():.2f}")
            print(f"  Mean: {self.data['SEDI'].mean():.2f}")
            print(f"  Std Dev: {self.data['SEDI'].std():.2f}")

            print("\n📝 Interpretation:")
            print("  Higher SEDI = Lower Deprivation (Better socioeconomic conditions)")
            print("  Lower SEDI = Higher Deprivation (Worse socioeconomic conditions)")
        else:
            raise ValueError("Unable to calculate SEDI - no domain scores available")

        return self.data


class DeprivationCategorizer:
    """
    Categorizes districts based on SEDI scores and creates rankings.
    """

    def __init__(self, data: pd.DataFrame):
        """
        Initialize DeprivationCategorizer.

        Parameters:
        -----------
        data : pd.DataFrame
            Dataset with SEDI scores
        """
        self.data = data.copy()

    def categorize(self, method: str = 'percentile') -> pd.DataFrame:
        """
        Categorize districts into deprivation levels.

        Parameters:
        -----------
        method : str
            Categorization method: 'percentile' or 'threshold'
            - percentile: Uses 33rd and 67th percentiles
            - threshold: Uses fixed thresholds (40, 60)

        Returns:
        --------
        pd.DataFrame
            Dataset with deprivation categories and ranks
        """
        print("\n" + "="*80)
        print("SECTION 1.6: DEPRIVATION CATEGORIZATION")
        print("="*80)

        if method == 'percentile':
            self._categorize_percentile()
        elif method == 'threshold':
            self._categorize_threshold()
        else:
            raise ValueError("Method must be 'percentile' or 'threshold'")

        # Create rankings (Rank 1 = lowest deprivation)
        self.data['SEDI_Rank'] = self.data['SEDI'].rank(ascending=False, method='min').astype(int)

        # Display summary
        self._display_categorization_summary()

        return self.data

    def _categorize_percentile(self):
        """Categorize using percentile method."""
        p33 = self.data['SEDI'].quantile(0.33)
        p67 = self.data['SEDI'].quantile(0.67)

        def assign_category(sedi):
            if sedi < p33:
                return 'High Deprivation'
            elif sedi < p67:
                return 'Medium Deprivation'
            else:
                return 'Low Deprivation'

        self.data['Deprivation_Category'] = self.data['SEDI'].apply(assign_category)

        print(f"\n📊 Percentile-Based Categorization:")
        print(f"  33rd percentile: {p33:.2f}")
        print(f"  67th percentile: {p67:.2f}")
        print(f"  High Deprivation: SEDI < {p33:.2f}")
        print(f"  Medium Deprivation: {p33:.2f} ≤ SEDI < {p67:.2f}")
        print(f"  Low Deprivation: SEDI ≥ {p67:.2f}")

    def _categorize_threshold(self, low_threshold: float = 40, high_threshold: float = 60):
        """Categorize using fixed thresholds."""
        def assign_category(sedi):
            if sedi < low_threshold:
                return 'High Deprivation'
            elif sedi < high_threshold:
                return 'Medium Deprivation'
            else:
                return 'Low Deprivation'

        self.data['Deprivation_Category'] = self.data['SEDI'].apply(assign_category)

        print(f"\n📊 Threshold-Based Categorization:")
        print(f"  High Deprivation: SEDI < {low_threshold}")
        print(f"  Medium Deprivation: {low_threshold} ≤ SEDI < {high_threshold}")
        print(f"  Low Deprivation: SEDI ≥ {high_threshold}")

    def _display_categorization_summary(self):
        """Display categorization summary."""
        print("\n" + "-"*80)
        print("CATEGORIZATION SUMMARY")
        print("-"*80)

        category_counts = self.data['Deprivation_Category'].value_counts()
        print("\nDistribution:")
        for category, count in category_counts.items():
            pct = (count / len(self.data)) * 100
            print(f"  {category}: {count} districts ({pct:.1f}%)")

        # Top 5 districts (lowest deprivation)
        print("\n🏆 Top 5 Districts (Lowest Deprivation):")
        top_5 = self.data.nsmallest(5, 'SEDI_Rank')[['District', 'SEDI', 'SEDI_Rank', 'Deprivation_Category']]
        for idx, row in top_5.iterrows():
            print(f"  {row['SEDI_Rank']}. {row['District']}: SEDI={row['SEDI']:.2f} ({row['Deprivation_Category']})")

        # Bottom 5 districts (highest deprivation)
        print("\n⚠ Bottom 5 Districts (Highest Deprivation):")
        bottom_5 = self.data.nlargest(5, 'SEDI_Rank')[['District', 'SEDI', 'SEDI_Rank', 'Deprivation_Category']]
        for idx, row in bottom_5.iterrows():
            print(f"  {row['SEDI_Rank']}. {row['District']}: SEDI={row['SEDI']:.2f} ({row['Deprivation_Category']})")


class OutputGenerator:
    """
    Generates output files: processed data, summary, metadata, visualizations, and data dictionary.
    """

    def __init__(self, data: pd.DataFrame, output_dir: str = 'output'):
        """
        Initialize OutputGenerator.

        Parameters:
        -----------
        data : pd.DataFrame
            Final processed dataset with SEDI
        output_dir : str
            Directory to save output files
        """
        self.data = data.copy()
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def generate_outputs(self, normalization_metadata: Dict):
        """
        Generate all output files.

        Parameters:
        -----------
        normalization_metadata : Dict
            Metadata from normalization process
        """
        print("\n" + "="*80)
        print("SECTION 1.7: OUTPUT GENERATION")
        print("="*80)

        # 1. Save processed data
        self._save_processed_data()

        # 2. Save SEDI summary
        self._save_sedi_summary()

        # 3. Save metadata
        self._save_metadata(normalization_metadata)

        # 4. Create visualizations
        self._create_visualizations()

        # 5. Generate data dictionary
        self._generate_data_dictionary()

        print("\n✅ All outputs generated successfully!")
        print(f"📁 Output directory: {self.output_dir.absolute()}")

    def _save_processed_data(self):
        """Save complete processed dataset."""
        filepath = self.output_dir / 'module1_processed_data.csv'
        self.data.to_csv(filepath, index=False)
        print(f"\n✓ Saved: module1_processed_data.csv ({len(self.data)} districts)")

    def _save_sedi_summary(self):
        """Save SEDI summary with key metrics."""
        summary_cols = ['District', 'SEDI', 'SEDI_Rank', 'Deprivation_Category']

        # Add domain scores if available
        domain_cols = [col for col in self.data.columns if col.endswith('_Score') and col != 'SEDI']
        summary_cols.extend(domain_cols)

        summary = self.data[summary_cols].copy()
        summary = summary.sort_values('SEDI_Rank')

        filepath = self.output_dir / 'module1_sedi_summary.csv'
        summary.to_csv(filepath, index=False)
        print(f"✓ Saved: module1_sedi_summary.csv")

    def _save_metadata(self, normalization_metadata: Dict):
        """Save comprehensive metadata."""
        from datetime import datetime

        metadata = {
            'project': 'District-Level Multidimensional Socioeconomic Deprivation Analysis in Karnataka',
            'module': 'Module 1 - Data Loading and SEDI Construction',
            'version': '2.0 (Simplified)',
            'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'data_sources': {
                'census_2011': {
                    'source': 'Census of India 2011',
                    'indicators': ['Literacy_Rate', 'Urbanization_Rate', 'Electricity_Access', 'Safe_Water_Access'],
                    'authority': 'Office of the Registrar General & Census Commissioner, India'
                },
                'niti_aayog_sdg': {
                    'source': 'NITI Aayog SDG India Index 2021-22',
                    'indicators': ['Education_Score', 'Health_Score', 'Infrastructure_Score'],
                    'authority': 'NITI Aayog, Government of India'
                },
                'economic_survey': {
                    'source': 'Economic Survey of Karnataka 2022-23',
                    'indicators': ['Per_Capita_Income', 'Unemployment_Rate', 'Healthcare_Facilities_Per_Lakh', 'Road_Density'],
                    'authority': 'Planning, Programme Monitoring and Statistics Department, Government of Karnataka'
                }
            },
            'methodology': {
                'normalization': 'Min-Max normalization with directionality consideration',
                'index_construction': 'Domain-based weighted average',
                'domain_weights': SEDIConstructor.DOMAIN_STRUCTURE
            },
            'normalization_metadata': normalization_metadata,
            'statistics': {
                'total_districts': int(len(self.data)),
                'sedi_range': [float(self.data['SEDI'].min()), float(self.data['SEDI'].max())],
                'sedi_mean': float(self.data['SEDI'].mean()),
                'sedi_std': float(self.data['SEDI'].std()),
                'category_distribution': self.data['Deprivation_Category'].value_counts().to_dict()
            }
        }

        filepath = self.output_dir / 'module1_metadata.json'
        with open(filepath, 'w') as f:
            json.dump(metadata, f, indent=2)
        print(f"✓ Saved: module1_metadata.json")

    def _create_visualizations(self):
        """Create SEDI visualizations."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Module 1: Socioeconomic Deprivation Index (SEDI) Analysis',
                     fontsize=16, fontweight='bold')

        # 1. SEDI distribution
        axes[0, 0].hist(self.data['SEDI'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
        axes[0, 0].axvline(self.data['SEDI'].mean(), color='red', linestyle='--',
                          label=f"Mean: {self.data['SEDI'].mean():.2f}")
        axes[0, 0].set_xlabel('SEDI Score', fontsize=11)
        axes[0, 0].set_ylabel('Frequency', fontsize=11)
        axes[0, 0].set_title('SEDI Distribution Across Districts', fontsize=12, fontweight='bold')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)

        # 2. Category distribution
        category_counts = self.data['Deprivation_Category'].value_counts()
        colors = {'Low Deprivation': '#2ecc71', 'Medium Deprivation': '#f39c12',
                 'High Deprivation': '#e74c3c'}
        axes[0, 1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
                      colors=[colors.get(cat, 'gray') for cat in category_counts.index],
                      startangle=90)
        axes[0, 1].set_title('Deprivation Category Distribution', fontsize=12, fontweight='bold')

        # 3. Top 10 and Bottom 10 districts
        top_10 = self.data.nsmallest(10, 'SEDI_Rank')[['District', 'SEDI']].sort_values('SEDI')
        bottom_10 = self.data.nlargest(10, 'SEDI_Rank')[['District', 'SEDI']].sort_values('SEDI')

        y_pos_top = np.arange(len(top_10))
        axes[1, 0].barh(y_pos_top, top_10['SEDI'], color='#2ecc71', alpha=0.7)
        axes[1, 0].set_yticks(y_pos_top)
        axes[1, 0].set_yticklabels(top_10['District'], fontsize=9)
        axes[1, 0].set_xlabel('SEDI Score', fontsize=11)
        axes[1, 0].set_title('Top 10 Districts (Lowest Deprivation)', fontsize=12, fontweight='bold')
        axes[1, 0].grid(alpha=0.3, axis='x')

        y_pos_bottom = np.arange(len(bottom_10))
        axes[1, 1].barh(y_pos_bottom, bottom_10['SEDI'], color='#e74c3c', alpha=0.7)
        axes[1, 1].set_yticks(y_pos_bottom)
        axes[1, 1].set_yticklabels(bottom_10['District'], fontsize=9)
        axes[1, 1].set_xlabel('SEDI Score', fontsize=11)
        axes[1, 1].set_title('Bottom 10 Districts (Highest Deprivation)', fontsize=12, fontweight='bold')
        axes[1, 1].grid(alpha=0.3, axis='x')

        plt.tight_layout()
        filepath = self.output_dir / 'module1_sedi_visualizations.png'
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✓ Saved: module1_sedi_visualizations.png")

    def _generate_data_dictionary(self):
        """Generate comprehensive data dictionary."""
        dictionary = """
================================================================================
MODULE 1: DATA DICTIONARY
================================================================================

PROJECT: District-Level Multidimensional Socioeconomic Deprivation Analysis
         in Karnataka, India

================================================================================
RAW INDICATORS (from Official Sources)
================================================================================

CENSUS OF INDIA 2011:
- Literacy_Rate: Percentage of literate population (age 7+) [0-100%]
- Urbanization_Rate: Percentage of population in urban areas [0-100%]
- Electricity_Access: Percentage of households with electricity [0-100%]
- Safe_Water_Access: Percentage of households with safe drinking water [0-100%]

NITI AAYOG SDG INDIA INDEX 2021-22:
- Education_Score: Composite score for SDG 4 (Quality Education) [0-100]
- Health_Score: Composite score for SDG 3 (Good Health and Well-being) [0-100]
- Infrastructure_Score: Composite score for SDG 9 & 11 (Infrastructure) [0-100]

ECONOMIC SURVEY OF KARNATAKA 2022-23:
- Per_Capita_Income: Average income per person in INR
- Unemployment_Rate: Percentage of unemployed workforce [0-100%]
- Healthcare_Facilities_Per_Lakh: Number of healthcare facilities per 100,000 population
- Road_Density: Road length in kilometers per square kilometer

================================================================================
NORMALIZED INDICATORS
================================================================================

All indicators normalized to [0, 1] scale using Min-Max normalization:
- Positive indicators (higher = better): X_norm = (X - X_min) / (X_max - X_min)
- Negative indicators (higher = worse): X_norm = 1 - [(X - X_min) / (X_max - X_min)]

Normalized variables have suffix: _norm
Example: Literacy_Rate_norm, Unemployment_Rate_norm

================================================================================
DOMAIN SCORES
================================================================================

Economic_Score: Average of normalized economic indicators (30% weight)
  - Per_Capita_Income_norm
  - Unemployment_Rate_norm

Education_Score: Average of normalized education indicators (25% weight)
  - Literacy_Rate_norm
  - Education_Score_norm

Health_Score: Average of normalized health indicators (20% weight)
  - Healthcare_Facilities_Per_Lakh_norm
  - Health_Score_norm

Infrastructure_Score: Average of normalized infrastructure indicators (25% weight)
  - Road_Density_norm
  - Electricity_Access_norm
  - Urbanization_Rate_norm
  - Infrastructure_Score_norm

================================================================================
COMPOSITE INDEX
================================================================================

SEDI (Socioeconomic Deprivation Index): [0-100 scale]
  Formula: SEDI = Σ(weight_d × domain_score_d) × 100

  Interpretation:
  - Higher SEDI = Lower Deprivation (Better socioeconomic conditions)
  - Lower SEDI = Higher Deprivation (Worse socioeconomic conditions)

================================================================================
CATEGORICAL VARIABLES
================================================================================

Deprivation_Category: Three-level categorization
  - Low Deprivation: Districts with high SEDI scores
  - Medium Deprivation: Districts with moderate SEDI scores
  - High Deprivation: Districts with low SEDI scores

SEDI_Rank: Rank based on SEDI score
  - Rank 1 = Lowest deprivation (highest SEDI)
  - Higher rank = Higher deprivation (lower SEDI)

================================================================================
METADATA
================================================================================

District: Official district name (standardized nomenclature)

Total variables in processed dataset: """ + str(len(self.data.columns)) + """
Total districts: """ + str(len(self.data)) + """

================================================================================
"""

        filepath = self.output_dir / 'module1_data_dictionary.txt'
        with open(filepath, 'w') as f:
            f.write(dictionary)
        print(f"✓ Saved: module1_data_dictionary.txt")


def run_module_1(filepath: str,
                 output_dir: str = 'output',
                 categorization_method: str = 'percentile') -> pd.DataFrame:
    """
    Execute complete Module 1 workflow.

    Parameters:
    -----------
    filepath : str
        Path to the input CSV file
    output_dir : str
        Directory for output files
    categorization_method : str
        Method for categorization: 'percentile' or 'threshold'

    Returns:
    --------
    pd.DataFrame
        Final processed dataset with SEDI
    """
    print("\n" + "="*80)
    print("MODULE 1: DATA LOADING AND SEDI CONSTRUCTION")
    print("="*80)
    print("Project: District-Level Multidimensional Socioeconomic Deprivation Analysis")
    print("Location: Karnataka, India")
    print("Version: 2.0 (Simplified)")
    print("="*80)

    # Section 1.1: Load data
    loader = DataLoader(filepath)
    data = loader.load_data()

    # Section 1.2: Validate data
    validator = DataValidator(data)
    data = validator.validate()

    # Section 1.3 & 1.4: Normalize indicators
    normalizer = IndicatorNormalizer(data)
    data = normalizer.normalize()

    # Section 1.5: Construct SEDI
    sedi_constructor = SEDIConstructor(data)
    data = sedi_constructor.construct_sedi()

    # Section 1.6: Categorize deprivation
    categorizer = DeprivationCategorizer(data)
    data = categorizer.categorize(method=categorization_method)

    # Section 1.7: Generate outputs
    output_gen = OutputGenerator(data, output_dir)
    output_gen.generate_outputs(normalizer.normalization_metadata)

    print("\n" + "="*80)
    print("✅ MODULE 1 COMPLETED SUCCESSFULLY")
    print("="*80)
    print(f"\nProcessed {len(data)} districts from Karnataka")
    print(f"SEDI range: {data['SEDI'].min():.2f} - {data['SEDI'].max():.2f}")
    print(f"Output files saved to: {output_dir}/")

    return data


# ================================================================================
# MAIN EXECUTION
# ================================================================================

if __name__ == "__main__":
    """
    Main execution block for Module 1.

    Usage:
    ------
    python module_1_simplified.py

    Requirements:
    -------------
    - Input file: karnataka_districts_dataset.csv (in current directory)
    - Python packages: pandas, numpy, matplotlib, seaborn

    Outputs:
    --------
    - module1_processed_data.csv: Complete processed dataset
    - module1_sedi_summary.csv: SEDI summary with rankings
    - module1_metadata.json: Comprehensive metadata
    - module1_sedi_visualizations.png: Visualization of SEDI results
    - module1_data_dictionary.txt: Variable documentation
    """

    try:
        # Configuration
        INPUT_FILE = '/content/drive/MyDrive/deprivation_analysis/karnataka_districts_dataset.csv'
        OUTPUT_DIR = '/content/drive/MyDrive/deprivation_analysis/output'
        CATEGORIZATION_METHOD = 'percentile'  # or 'threshold'

        # Run Module 1
        processed_data = run_module_1(
            filepath=INPUT_FILE,
            output_dir=OUTPUT_DIR,
            categorization_method=CATEGORIZATION_METHOD
        )

        # Display final summary
        print("\n" + "="*80)
        print("FINAL DATASET PREVIEW")
        print("="*80)
        print(processed_data[['District', 'SEDI', 'SEDI_Rank', 'Deprivation_Category']].head(10))

    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

MODULE 1: DATA LOADING AND SEDI CONSTRUCTION
Project: District-Level Multidimensional Socioeconomic Deprivation Analysis
Location: Karnataka, India
Version: 2.0 (Simplified)

SECTION 1.1: DATA LOADING
✓ Successfully loaded data from: /content/drive/MyDrive/deprivation_analysis/karnataka_districts_dataset.csv
✓ Dataset shape: 31 districts × 12 variables
✓ District names standardized to official nomenclature

First 5 districts:
          District  Literacy_Rate  Urbanization_Rate  Electricity_Access  Safe_Water_Access  Per_Capita_Income  \
0         Bagalkot           70.5               25.3                89.2               78.5              98500   
1          Ballari           67.4               39.8                91.5               82.1             125000   
2         Belagavi           73.5               28.7                90.8               80.3       